# 07. Service Mesh Security

## Background

In microservice architectures, services communicate over the network. Traditional TLS secures the wire but requires each service to manage its own certificates. Service meshes like Istio and Linkerd solve this with a sidecar proxy model: a lightweight proxy (Envoy) is injected alongside each service container, and the mesh handles mutual TLS (mTLS), service identity, and traffic policy automatically.

SPIFFE (Secure Production Identity Framework For Everyone) and SPIRE (SPIFFE Runtime Environment) provide the underlying identity layer: each workload gets a SPIFFE Verifiable Identity Document (SVID) — a short-lived X.509 certificate — attesting to its identity. No passwords, no API keys, just cryptographic workload identity.

## What You'll Learn

- mTLS: both client and server present certificates, enabling mutual authentication
- SPIFFE ID format: `spiffe://trust-domain/path`
- SVID lifecycle: issuance, rotation, TTL
- Service-to-service authorization using SPIFFE identities
- Simulating mTLS verification and SPIFFE identity validation in Python

## Why This Matters

Service meshes are the zero trust implementation layer for microservices. Without mTLS, a compromised container inside the cluster can impersonate any service. SPIFFE/SPIRE are now CNCF graduated projects and the standard workload identity layer for Kubernetes, AWS EKS, and GCP GKE.

In [ ]:
import hashlib, hmac, secrets, time, json
from dataclasses import dataclass, field
from typing import Optional
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.style.use('seaborn-v0_8-whitegrid')
print('Service mesh security demo ready')


## 1. SPIFFE Identity and SVID

In [ ]:
@dataclass
class SVID:
    spiffe_id:    str    # spiffe://trust-domain/ns/default/sa/frontend
    serial:       str
    issued_at:    float
    expires_at:   float
    trust_domain: str
    fingerprint:  str    # sha256 of simulated cert bytes

    @property
    def is_valid(self):
        return time.time() < self.expires_at

    @property
    def workload_path(self):
        return self.spiffe_id.split(self.trust_domain)[-1].lstrip('/')


class SPIREWorkloadAPI:
    def __init__(self, trust_domain='cluster.local', ttl=3600):
        self.trust_domain = trust_domain
        self.ttl = ttl
        self._ca_key = secrets.token_bytes(32)   # simulated CA

    def issue_svid(self, workload_path):
        spiffe_id = f'spiffe://{self.trust_domain}/{workload_path}'
        now = time.time()
        serial = secrets.token_hex(8)
        cert_bytes = f'{spiffe_id}:{serial}:{now}'.encode()
        fingerprint = hmac.new(self._ca_key, cert_bytes, hashlib.sha256).hexdigest()
        return SVID(spiffe_id=spiffe_id, serial=serial, issued_at=now,
                    expires_at=now + self.ttl, trust_domain=self.trust_domain,
                    fingerprint=fingerprint)

    def verify_svid(self, svid):
        if not svid.is_valid:
            return False, 'SVID expired'
        if not svid.spiffe_id.startswith(f'spiffe://{self.trust_domain}/'):
            return False, 'Wrong trust domain'
        cert_bytes = f'{svid.spiffe_id}:{svid.serial}:{svid.issued_at}'.encode()
        expected   = hmac.new(self._ca_key, cert_bytes, hashlib.sha256).hexdigest()
        if not hmac.compare_digest(expected, svid.fingerprint):
            return False, 'Certificate verification failed'
        return True, 'Valid'


spire = SPIREWorkloadAPI(trust_domain='prod.cluster.local', ttl=3600)
services = ['ns/default/sa/frontend', 'ns/default/sa/api-gateway',
            'ns/ml/sa/inference-svc', 'ns/ml/sa/feature-store']
svids = {svc: spire.issue_svid(svc) for svc in services}
for svc, svid in svids.items():
    ok, reason = spire.verify_svid(svid)
    print(f'[{"VALID" if ok else "FAIL "}] {svid.spiffe_id}')


## 2. mTLS Authorization Policy

In [ ]:
MTLS_POLICY = {
    'ns/default/sa/api-gateway':  ['ns/default/sa/frontend'],
    'ns/ml/sa/inference-svc':     ['ns/default/sa/api-gateway','ns/ml/sa/feature-store'],
    'ns/ml/sa/feature-store':     ['ns/ml/sa/inference-svc'],
}

def mtls_authorize(client_svid, server_path, spire_instance):
    ok, reason = spire_instance.verify_svid(client_svid)
    if not ok:
        return False, f'Client SVID invalid: {reason}'
    client_path = client_svid.workload_path
    allowed_clients = MTLS_POLICY.get(server_path, [])
    if client_path not in allowed_clients:
        return False, f'Client {client_path} not authorized to call {server_path}'
    return True, f'mTLS authorized'


print('mTLS service-to-service authorization:')
calls = [
    ('ns/default/sa/frontend',    'ns/default/sa/api-gateway'),
    ('ns/default/sa/api-gateway', 'ns/ml/sa/inference-svc'),
    ('ns/default/sa/frontend',    'ns/ml/sa/inference-svc'),  # not allowed
    ('ns/ml/sa/inference-svc',    'ns/ml/sa/feature-store'),
    ('ns/default/sa/frontend',    'ns/ml/sa/feature-store'),  # not allowed
]
for caller, callee in calls:
    ok, reason = mtls_authorize(svids[caller], callee, spire)
    print(f'  [{"ALLOW" if ok else "DENY "}] {caller.split("/")[-1]} -> {callee.split("/")[-1]}  ({reason})')


## 3. Service Mesh Architecture Diagram

In [ ]:
import networkx as nx

G = nx.DiGraph()
nodes = ['frontend','api-gateway','inference-svc','feature-store','[BLOCKED]']
G.add_edges_from([
    ('frontend','api-gateway'),
    ('api-gateway','inference-svc'),
    ('inference-svc','feature-store'),
])
pos = {'frontend':(0,1),'api-gateway':(1,1),'inference-svc':(2,1),'feature-store':(3,1)}

fig, ax = plt.subplots(figsize=(10,4))
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=2500, node_color='#E3F2FD', edgecolors='#1565C0', linewidths=2)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=9, font_weight='bold')
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowsize=25, width=2,
                       edge_color='#4CAF50', connectionstyle='arc3,rad=0.1')
# Blocked path
ax.annotate('', xy=(2.0,0.85), xytext=(0,0.85),
            arrowprops=dict(arrowstyle='->', color='red', lw=2, linestyle='dashed'))
ax.text(1.0, 0.75, 'BLOCKED by mTLS policy\n(frontend cannot reach inference-svc directly)',
        ha='center', color='red', fontsize=9)
for node, (x,y) in pos.items():
    ax.text(x, y-0.18, 'SVID: spiffe://.../' + node, ha='center', fontsize=7, color='gray')
ax.set_xlim(-0.5, 3.5); ax.set_ylim(0.4, 1.4); ax.axis('off')
ax.set_title('Service Mesh mTLS — SPIFFE Identity + Authorization Policy', fontsize=12)
plt.tight_layout()
plt.savefig('07_service_mesh.png', dpi=110, bbox_inches='tight')
plt.show()
